# TP · Identidad y emoción de actores argentinos — Grupo 6
**IAyAA II · Licenciatura en Ciencia de Datos · UCA Rosario · 2026**

Martín Masuelli · Francisco Miles · Tomás Lizazu · Bautista Verna · Juan Diego Ancarani · Manuel Lafosse

Dos tareas sobre el dataset del curso (recortes de rostro 224×224):
- **Identidad:** ¿quién es? (20 actores en la versión actual)
- **Emoción:** ¿qué gesto hace? (alegría, enojo, neutral, sorpresa, tristeza)

**Recorrido** (cada paso responde a una pregunta y cada decisión está justificada; el registro completo está en `docs/BITACORA.md` del repo):

| # | Sección | Pregunta que responde |
|---|---|---|
| 1 | Entorno | ¿Cómo se reproduce exactamente este notebook? |
| 2 | Datos | ¿Qué hay en el dataset y qué le falta? |
| 3 | Partición | ¿Cómo evitamos que el modelo "reconozca el video" en vez del gesto? |
| 4 | Baselines | ¿Contra qué piso medimos? |
| 5 | Acta de límites | ¿Por qué hace falta una red convolucional? |
| 6 | Pipeline y aumentación | ¿Cómo se alimenta la red y qué invarianzas le enseñamos? |
| 7 | CNN propia | ¿Cuánto rinde una red entrenada desde cero con pocos datos? |
| 8 | ResNet50 (transferencia) | ¿Cuánto aporta lo aprendido en ImageNet y el ajuste fino? |
| 9 | Evaluación final en test | Resultado honesto, evaluado **una sola vez** |
| 10 | Interpretación | ¿El modelo mira la cara o el fondo? |
| 11 | Conclusiones y limitaciones | |

> **Reglas que respeta este notebook:** el Drive del curso se monta en **solo lectura**; la partición es **por video**; toda decisión se toma contra **validación** y el test se evalúa una única vez al final; semillas fijas.

## 1. Entorno reproducible

Tres cosas fijan la reproducibilidad:
1. **El código** sale del repositorio del grupo (`github.com/FMiles-2006/tp-emociones-g6`): toda la lógica vive en el paquete `tp_emociones` y el notebook solo la orquesta. Se imprime el *commit* exacto usado.
2. **Los datos** salen del `DATASET_FINAL` del Drive del curso, montado en **solo lectura** (`readonly=True`) y copiado a disco local. Se imprime la **huella** (hash) de la versión del dataset: si alguien agrega imágenes, la huella cambia y los números quedan atados a su versión.
3. **El azar** se controla con una semilla única (`config.SEED = 42`) fijada en Python, NumPy y TensorFlow.

> Requisito: tener en "Mi unidad" un acceso directo a la carpeta `CARPETA COMPARTIDA PARA TODOS` (clic derecho → Organizar → Agregar acceso directo → Mi unidad). Entorno de ejecución recomendado: **GPU T4**.

In [ ]:
# 1.1 Código del grupo (versión fija del repositorio)
import os, sys, subprocess
REPO_URL = "https://github.com/FMiles-2006/tp-emociones-g6"
REPO_DIR = "/content/tp-emociones-g6"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Código del grupo, commit:", COMMIT)

In [ ]:
# 1.2 Datos: Drive del curso en SOLO LECTURA + copia local (más rápida y sin riesgo de tocar el original)
import glob, shutil
from google.colab import drive
drive.mount("/content/drive", readonly=True)

candidatas = glob.glob("/content/drive/MyDrive/**/CARPETA COMPARTIDA PARA TODOS/DATASET_FINAL", recursive=True) \
           + glob.glob("/content/drive/MyDrive/CARPETA COMPARTIDA PARA TODOS/DATASET_FINAL")
assert candidatas, "No se encontró DATASET_FINAL: agregá el acceso directo a la carpeta del curso en 'Mi unidad'."
ORIGEN = candidatas[0]
DATASET = "/content/datos/DATASET_FINAL"
if not os.path.exists(DATASET):
    shutil.copytree(ORIGEN, DATASET, ignore=shutil.ignore_patterns("Metricas_Dataset"))
print("Origen (solo lectura):", ORIGEN)
print("Copia de trabajo:     ", DATASET, "→", len(os.listdir(f"{DATASET}/Imagenes_Finales")), "imágenes")

In [ ]:
# 1.3 Librerías, semillas y versiones
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, keras, sklearn
from IPython.display import display

from tp_emociones import config, datos, particion, baselines, pipeline, modelos, evaluacion, experimentos

config.fijar_semillas()
sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)
print(f"TensorFlow {tf.__version__} · Keras {keras.__version__} · scikit-learn {sklearn.__version__} · NumPy {np.__version__}")
print("GPU:", tf.config.list_physical_devices("GPU") or "NO (activar GPU en Entorno de ejecución)")
print("Semilla:", config.SEED)

## 2. Los datos

`datos.construir_registro` lee el CSV del curso y arma una fila por imagen utilizable. Resuelve tres problemas que encontramos al auditar el dataset:
- **Nombres inconsistentes** (tildes, mayúsculas, `ñ` codificada como `#U00f1` por Drive): se cruzan por una clave normalizada.
- **Identidades escritas distinto** (`tini` / `tini_stoessel`): se unifican con `config.MAPA_ACTORES`.
- **El CSV no tiene el video de origen**, indispensable para la partición. Se recupera de los registros que subió cada grupo y, si no está ahí, del nombre de archivo (`actor__video__f123.jpg`, `Cande1_frame_10.jpg`). Último recurso conservador: el nombre sin números.

In [ ]:
df = datos.construir_registro(DATASET)
HUELLA = df.attrs["huella"]
df.head()

In [ ]:
# Distribución actor × emoción (tabla del enunciado) y gráfico
tabla = datos.tabla_distribucion(df)
display(tabla)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 2]})
df["emocion"].value_counts().reindex(config.EMOCIONES).plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Imágenes por emoción"); axes[0].set_ylabel("imágenes")
df["actor"].value_counts().plot.bar(ax=axes[1], color="slategray")
axes[1].set_title("Imágenes por actor"); plt.tight_layout(); plt.show()

In [ ]:
# Grilla de ejemplos: una fila por emoción
from PIL import Image
rng = np.random.default_rng(config.SEED)
fig, axes = plt.subplots(len(config.EMOCIONES), 8, figsize=(14, 9))
for f, emo in enumerate(config.EMOCIONES):
    muestra = df[df["emocion"] == emo].sample(8, random_state=config.SEED)
    for c, (_, fila) in enumerate(muestra.iterrows()):
        axes[f, c].imshow(Image.open(fila["ruta"])); axes[f, c].axis("off")
        axes[f, c].set_title(fila["actor"].split("_")[0], fontsize=7)
    axes[f, 0].text(-40, 112, emo, rotation=90, va="center", fontsize=11)
plt.suptitle("Ejemplos por emoción"); plt.tight_layout(); plt.show()

**Lectura del dataset (para la ficha):**
- El desbalance de emociones es moderado: *neutral* es la más frecuente y *tristeza* la más escasa (≈12 %). Por eso reportamos **F1 macro** además de accuracy y entrenamos con **pesos por clase**.
- Identidad: la mayoría de los actores tiene ~100 imágenes, pero Francella (419) y Tini (243) están sobrerrepresentados porque aparecen en dos grupos.
- **Limitaciones conocidas:** Pachu Peña (122 imágenes del G6) no entró al dataset del curso por un problema con la "ñ" del nombre de archivo; Rodrigo de la Serna no tiene imágenes subidas; los grupos 2, 5 y 9 no aportaron. Los recortes del G6 son centrales (la cara ocupa menos del cuadro que en otros grupos).
- El **techo realista** para emoción es el acuerdo entre etiquetadores: en nuestro control cruzado del TP1 fue **42,6 %** (6 clases). Ningún modelo debería "superar" de forma creíble lo que los humanos acuerdan entre sí.

## 3. Partición por video (la regla de oro)

Dos cuadros del mismo plano son casi idénticos: en la auditoría encontramos ~1.500 pares de imágenes casi duplicadas dentro de un mismo video. Si se reparten entre train y test, el modelo **reconoce el plano y no el gesto**, y el resultado se infla.

Reglas implementadas en `particion.py` (detalle en BITACORA D-004):
- **R1 (sin excepciones):** ningún video de train aparece en validación ni en test. Se verifica automáticamente.
- **R2:** se parte por actor, para que los 20 actores estén en train (indispensable para identidad) y también en val y test.
- **R3:** por actor se eligen los videos de train buscando ~70 % de sus imágenes; el resto se reparte entre val y test **por video**.
- **R4 (concesión documentada):** en actores con muy pocos videos o muy desparejos, val y test comparten **un** video cortado en dos mitades contiguas. Train nunca comparte.
- Es **determinista**: mismo dataset + misma semilla ⇒ misma partición.

In [ ]:
df["split"] = particion.particionar(df)
particion.verificar_sin_fugas(df, df["split"])
print("✔ Sin fugas: ningún video de train aparece en validación o test.\n")
display(particion.resumen_particion(df, df["split"]))
print("\nActores donde se aplicó R4 (val y test comparten un video):")
for aviso in df["split"].attrs.get("avisos", []):
    print("  -", aviso)

In [ ]:
# Imágenes por actor y split
display(pd.crosstab(df["actor"], df["split"])[["train", "val", "test"]].assign(
    pct_train=lambda t: (100 * t["train"] / t.sum(axis=1)).round(0)))

## 4. Baselines: el piso contra el que se mide todo

Ninguna red se justifica sin un baseline (NB01). Medimos tres, **en validación** (test queda reservado):
- **Trivial:** predice siempre la clase más frecuente de train.
- **Random Forest y MLP de scikit-learn sobre píxeles**: imagen en gris reducida a 64×64 → 4.096 columnas. Se reduce porque con 224×224×3 = 150.528 columnas y ~1.900 imágenes no hay de dónde generalizar.

In [ ]:
X = baselines.cargar_pixeles(df["ruta"])
m_tr, m_va = (df["split"] == "train").to_numpy(), (df["split"] == "val").to_numpy()

resultados_baseline = {}
for tarea in ("actor", "emocion"):
    t = baselines.evaluar_baselines(X[m_tr], df.loc[m_tr, tarea], X[m_va], df.loc[m_va, tarea])
    resultados_baseline[tarea] = t
    print(f"\n=== Tarea: {tarea}  (VALIDACIÓN)"); display(t)

## 5. Acta de límites (NB02) sobre nuestro dataset

Las dos mediciones del NB02, repetidas con nuestros datos:
1. **Costo:** al agrandar el MLP, el tiempo crece mucho más rápido que el beneficio.
2. **Permutación de píxeles:** mezclamos los píxeles de *todas* las imágenes con una misma permutación fija. Para un humano quedan irreconocibles. Si el MLP rinde igual, es que **nunca usó la estructura espacial** de la cara: la trata como una tabla de números sueltos.

In [ ]:
costo = baselines.medir_costo_mlp(X[m_tr], df.loc[m_tr, "actor"], X[m_va], df.loc[m_va, "actor"])
display(costo)

perm_mlp = {t: baselines.experimento_permutacion_mlp(X[m_tr], df.loc[m_tr, t], X[m_va], df.loc[m_va, t]) for t in ("actor", "emocion")}
display(pd.DataFrame(perm_mlp).T.rename(columns={"original": "acc original", "permutada": "acc píxeles permutados"}))

**Acta:** el MLP rinde prácticamente igual con los píxeles destruidos ⇒ no ve una imagen, ve 4.096 números. Requisito para la herramienta siguiente: **capas convolucionales**, que comparten pesos sobre vecindades de píxeles y por lo tanto *sí* dependen de la estructura espacial (lo verificamos en la sección 7 repitiendo el experimento con la CNN). Además, el costo crece sin control en CPU ⇒ **GPU** y **callbacks** (early stopping) para controlar el entrenamiento.

## 6. Pipeline `tf.data` y aumentación

Patrón del curso: `map(leer) → cache → shuffle (solo train) → batch → prefetch`. Las imágenes se cachean como `uint8` (≈400 MB) y llegan al modelo en [0, 255].

**Aumentación como capas del modelo** (clase 17): solo actúa al entrenar, y el modelo guardado ya lleva su preprocesamiento. Cada transformación declara una invarianza válida **para rostros**:

| Transformación | ¿Válida? | Argumento |
|---|---|---|
| Flip horizontal | Sí | Una cara espejada es la misma persona con el mismo gesto |
| Rotación ±8° | Sí | Las cabezas se inclinan en los videos |
| Zoom ±10 % / traslación ±5 % | Sí | Los recortes no están perfectamente centrados |
| Brillo / contraste ±15 % | Sí | Las fuentes tienen iluminación muy distinta |
| Flip vertical | **No** | Produce caras imposibles |
| Rotaciones grandes / recortes agresivos | **No** | Fabrican poses imposibles o tapan cejas y boca, que son la señal de la emoción |

In [ ]:
# Ejemplos: una imagen de train y 7 versiones aumentadas
aug = modelos.capas_aumentacion()
img = np.asarray(Image.open(df.loc[df["split"] == "train", "ruta"].iloc[0]).convert("RGB"), dtype="float32")
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
axes[0].imshow(img.astype("uint8")); axes[0].set_title("original")
for k in range(1, 8):
    axes[k].imshow(np.clip(aug(img[None], training=True)[0].numpy(), 0, 255).astype("uint8")); axes[k].set_title(f"aumentada {k}")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## 7. CNN propia desde cero

Anatomía del NB11 adaptada a 224×224: 4 bloques `[Conv 3×3 → Conv 3×3 → BatchNorm → MaxPool]` con filtros 32 → 64 → 128 → 256 (cada bloque reduce la resolución a la mitad y duplica los filtros: menos *dónde*, más *qué*), cierre con `GlobalAveragePooling → Dense(128) → Dropout(0,5) → softmax`.

Decisiones:
- **GlobalAveragePooling en vez de Flatten:** con Flatten la densa recibiría ~50.000 entradas (millones de parámetros para ~1.900 imágenes = memorizar).
- **Regularización:** aumentación + Dropout + L2 (1e-4) + early stopping sobre `val_loss` con restauración de los mejores pesos (NB Sobreajuste).
- **Pesos por clase** para el desbalance.
- Expectativa (NB14): con ~90 imágenes por actor, una red desde cero queda lejos de la transferencia. Sirve como **experimento de control** (así lo propusimos en el TP1).

In [ ]:
exps = {}
for tarea in ("actor", "emocion"):
    exps[("cnn", tarea)] = experimentos.entrenar_cnn_propia(df, tarea)
    e = exps[("cnn", tarea)]
    evaluacion.graficar_curvas(e.historias, f"CNN propia — {tarea}"); plt.show()
    print(f"{tarea}: VAL acc={e.val['accuracy']:.3f}  F1m={e.val['f1_macro']:.3f}  · parámetros={e.params['total']:,}  · épocas={e.params['epocas_reales']}")

### 7.1 La revancha de la permutación (NB11)
Repetimos el experimento de la sección 5 con la CNN: misma arquitectura, misma semilla, pero entrenada y validada con los píxeles permutados. Si la CNN **sí** usa la estructura espacial, su desempeño se tiene que desplomar (al revés que el MLP).

In [ ]:
perm = baselines.permutacion_fija(config.IMG_SIZE * config.IMG_SIZE)
_, clases_actor, ys_actor = pipeline.datasets_por_split(df, "actor")
ds_tr_p = pipeline.crear_dataset(df.loc[df["split"] == "train", "ruta"], ys_actor["train"], True, permutacion=perm)
ds_va_p = pipeline.crear_dataset(df.loc[df["split"] == "val", "ruta"], ys_actor["val"], False, permutacion=perm)
cnn_perm = modelos.cnn_propia(len(clases_actor), aumentar=False, nombre="cnn_permutada")
modelos.entrenar(cnn_perm, ds_tr_p, ds_va_p, lr=1e-3, epocas=60, paciencia=8,
                 pesos_clase=pipeline.pesos_por_clase(ys_actor["train"], len(clases_actor)), verbose=0)
acc_perm = evaluacion.metricas(ys_actor["val"], evaluacion.predecir(cnn_perm, ds_va_p).argmax(1))["accuracy"]
print(f"Identidad — CNN con imágenes originales: {exps[('cnn','actor')].val['accuracy']:.3f}  |  con píxeles permutados: {acc_perm:.3f}")
print(f"(MLP: original {perm_mlp['actor']['original']:.3f} | permutado {perm_mlp['actor']['permutada']:.3f})")

## 8. Transferencia: ResNet50 preentrenada en ImageNet

Por qué ResNet50 (propuesta del TP1): nuestros recortes ya son 224×224×3, su **entrada nativa** (no hay que inventar ni perder información), y con ~90 imágenes por clase no alcanza para aprender filtros desde cero. Las primeras capas de ResNet50 detectan bordes y texturas genéricas que sirven tal cual; las últimas (conv5) codifican semántica de ImageNet y son las que hay que adaptar a rostros.

Secuencia del NB15 (el orden **es** el contenido):
- **Fase 1:** base congelada, solo se entrena la cabeza nueva (`GAP → Dropout(0,3) → softmax`) con lr 1e-3. Una cabeza aleatoria produce gradientes grandes que, con la base libre, destruirían lo aprendido.
- **Fase 2:** se descongela solo el bloque **conv5**, con lr **1e-5** (100 veces menor) y BatchNorm en modo inferencia (`training=False`). Recompilar es obligatorio tras cambiar `trainable`.
- Cada base exige **su** `preprocess_input` (NB13): va como capa dentro del modelo.

In [ ]:
for tarea in ("actor", "emocion"):
    exps[("resnet", tarea)] = experimentos.entrenar_resnet50(df, tarea)
    e = exps[("resnet", tarea)]
    evaluacion.graficar_curvas(e.historias, f"ResNet50 — {tarea}"); plt.show()
    print(f"{tarea}: VAL fase 1 acc={e.params['val_f1']['accuracy']:.3f} F1m={e.params['val_f1']['f1_macro']:.3f}"
          f"  →  fase 2 acc={e.val['accuracy']:.3f} F1m={e.val['f1_macro']:.3f}")
    print(f"   parámetros entrenables: fase 1 = {e.params['entrenables_f1']:,} · fase 2 = {e.params['entrenables_f2']:,} (de {e.params['total']:,})")

### 8.1 Comparación en validación (acá se toman las decisiones)

In [ ]:
filas = []
for tarea in ("actor", "emocion"):
    base_t = resultados_baseline[tarea]
    for nombre, fila in base_t.iterrows():
        filas.append({"tarea": tarea, "modelo": nombre, "accuracy": fila["accuracy"], "f1_macro": fila["f1_macro"]})
    filas.append({"tarea": tarea, "modelo": "ResNet50 fase 1 (congelada)", **exps[("resnet", tarea)].params["val_f1"]})
    for clave in (("cnn", tarea), ("resnet", tarea)):
        filas.append({"tarea": tarea, "modelo": exps[clave].nombre, **exps[clave].val})
comparacion_val = pd.DataFrame(filas).set_index(["tarea", "modelo"])
display(comparacion_val.round(3))

## 9. Evaluación final en test — una sola vez

Todas las decisiones anteriores se tomaron con validación. Ahora, y solo ahora, se evalúan en test los modelos finales (CNN propia y ResNet50 fase 2, para cada tarea). Esta celda no se vuelve a correr para "probar otra cosa": si algo cambia, cambia en validación y se documenta.

In [ ]:
resultados_test = {}
for clave, e in exps.items():
    resultados_test[clave] = experimentos.evaluar_en_test(e, df, mostrar=False)

tabla_test = pd.DataFrame([{"tarea": t, "modelo": exps[(m, t)].nombre, **resultados_test[(m, t)]["metricas"]}
                           for (m, t) in exps]).set_index(["tarea", "modelo"])
display(tabla_test.round(3))
print("Huella del dataset:", HUELLA, "· commit del código:", COMMIT, "· semilla:", config.SEED)

In [ ]:
# Matrices de confusión y reporte por clase del mejor modelo de cada tarea (ResNet50)
for tarea in ("actor", "emocion"):
    e, r = exps[("resnet", tarea)], resultados_test[("resnet", tarea)]
    evaluacion.reporte(e.ys["test"], r["prob"], e.clases, f"ResNet50 [{tarea}] TEST")
    evaluacion.graficar_matriz_confusion(e.ys["test"], r["prob"], e.clases, f"ResNet50 — {tarea} (test, normalizada por fila)"); plt.show()
    print("Pares más confundidos:"); display(evaluacion.pares_mas_confundidos(e.ys["test"], r["prob"], e.clases))

In [ ]:
# Errores con mayor confianza
for tarea in ("actor", "emocion"):
    e, r = exps[("resnet", tarea)], resultados_test[("resnet", tarea)]
    evaluacion.mostrar_peores_errores(r["rutas"], e.ys["test"], r["prob"], e.clases, titulo=f"Peores errores — {tarea}"); plt.show()

## 10. Interpretación: ¿el modelo mira la cara?

Mapa de oclusión (NB12): se tapa cada región con un parche gris y se mide cuánto cae la probabilidad de la clase correcta. En rojo, las regiones decisivas. Es la prueba del riesgo que declaramos en el TP1: como cada actor sale de pocas fuentes, un modelo de identidad podría estar reconociendo **el fondo o la serie**. Para emoción, esperamos ver la evidencia en **cejas, ojos y boca**.

In [ ]:
for tarea in ("actor", "emocion"):
    e, r = exps[("resnet", tarea)], resultados_test[("resnet", tarea)]
    bien = np.where(r["prob"].argmax(1) == e.ys["test"])[0]
    elegidas = np.random.default_rng(config.SEED).choice(bien, size=min(5, len(bien)), replace=False)
    evaluacion.graficar_oclusiones(e.modelo, r["rutas"][elegidas], e.ys["test"][elegidas], e.clases, n=len(elegidas),
                                   titulo=f"Oclusión — ResNet50 {tarea}"); plt.show()

## 11. Conclusiones y limitaciones

_Se completa con los números de las secciones 8 y 9 de esta corrida (huella y commit impresos arriba)._

**Declaración de uso de IA generativa:** se usó un asistente de IA (Claude) para auditar el dataset, proponer la estructura del código y redactar borradores de la documentación. Todas las decisiones fueron revisadas por el grupo y cualquier integrante puede explicar cada celda.